# Designing Powertrain Systems with Z3

In powertrain design, we often need to achieve a specific **Gear Ratio ($G_{total}$)** to transition power from a high-speed motor to a lower-speed output shaft.

The challenge is that the number of teeth on each gear ($N$) must be an **integer**, and there are physical constraints:

1. **Undercutting:** To avoid interference, gears usually need a minimum number of teeth (e.g., $N \ge 12$).
2. **Space Constraints:** Gears cannot be infinitely large (e.g., $N \le 100$).
3. **Exact Ratios:** For precision machinery, we often need an exact rational ratio that is difficult to find by hand.

In this notebook, we will
look at SMT solvers and see how they can be used to formalize and solve design problems in powertrain systems. We will focus on the design of a **Compound Gear Train**, a fundamental component in power transmission shafts and gear drives.


**Instructions:**
1. To get started, click on File on the top left and click "Save a copy in Drive."
This will give you an editable version of this document that you can use.
2. If you press `CMD`+`Enter` it runs the cell, and if you press `Shift`+`Enter` it runs the cell and goes to the next one.
3. Make sure you run all cells as you go through the notebook; some cells will not work properly unless the previous one
has been run too.
4. If you disconnect or are inactive for some time you should run all of the cells again.

## 0. Preliminaries (you should run this cell but there is no need to read it)

In [ ]:
!pip install z3-solver
!pip install git+https://github.com/crrivero/FormalMethodsTasting.git#subdirectory=core
from z3 import *
from tofmcore import plot_gears
import networkx as nx
import itertools
from IPython.display import clear_output
clear_output()

## Encoding constraints in Z3

The goal of this notebook is to teach you about formal methods;
particularly, how you can use existing formal verification tools
(in this case, Z3) to analyze and solve your own problems.
Before we get started, let's look at some basic things we can do with Z3.

### Reals

Let's use Z3 to solve problems involving reals. Let's start with something simple: find $x$ such that

$$2x + 5 = 15$$

In [ ]:
# Initialize variables

x = Int('x') # declairing that x is an integer named 'x'

# Initialize Z3 solver
s = Solver()

s.add( 2*x + 5 == 15 ) # add the equation

print(s)
print(s.check())
print(s.model())

Now let's try to check whether that's the only solution. We can do this by adding the following constraint to the solver:

$$x \not= 5$$

If the solver returns "**unsat**" then $x=5$ is the only solution.
Try it yourself by completing the code in the cell below.

In [ ]:
s.add( x == 5 ) # REPLACE THIS LINE
s.check()

**Note that if we were to run s.model() now we would get an error.**

## Modeling and Solving the Powertain Design





#### 1. Create a fresh solver


In [ ]:
powertrain_solver = Solver()

#### 2. Declare Variables


In [ ]:
# Define integer variables for the number of teeth
N1 = Int('N1')
N2 = Int('N2')
N3 = Int('N3')
N4 = Int('N4')

#### 3. Add Constraints (Method of Joints)

In [ ]:
# 1. Minimum teeth to avoid undercut (standard engineering practice)
min_teeth = 12
powertrain_solver.add(N1 >= min_teeth, N2 >= min_teeth, N3 >= min_teeth, N4 >= min_teeth)

# 2. Maximum teeth due to housing size constraints
max_teeth = 80
powertrain_solver.add(N1 <= max_teeth, N2 <= max_teeth, N3 <= max_teeth, N4 <= max_teeth)

# 3. The Gear Ratio Constraint
# Since Z3 handles integers and reals differently, we rearrange:
# (N2 / N1) * (N4 / N3) = 12  =>  N2 * N4 = 12 * N1 * N3
powertrain_solver.add(N2 * N4 == 12 * N1 * N3)

# 4. Avoiding "Idler" redundancy (ensure both stages contribute to reduction)
powertrain_solver.add(N2 > N1)
powertrain_solver.add(N4 > N3)

#### 4. Solve the System


In [ ]:
# Check if solution exists
print( powertrain_solver.check() )

In [ ]:
# View solution
solution = powertrain_solver.model()
print( solution )

Here, we will visualize the **relative sizes of the gears** in the powertrain layout to ensure the design looks physically feasible.

In [ ]:
# Run the visualization with the solver's results
n1, n2, n3, n4 = [solution[v].as_long() for v in [N1, N2, N3, N4]]
plot_gears(n1, n2, n3, n4)


### Congratulations! You just used an SMT solver to solve trusses!


####If you'd like to continue your Z3 journey, you can start with this guide to learn more:
https://ericpony.github.io/z3py-tutorial/guide-examples.htm